This notebook accompanies the contribution "Canonicity, Gender and Time Period. An Empirical Reconstruction of an Academic Canon".

# Preparation

## Import

In [223]:
import pandas as pd
from scipy.stats import zscore, chi2_contingency
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objs as go
import json
import re
from tqdm import tqdm

In [224]:
# import data file with information about all indicators.
data = pd.read_csv("../data/data.csv", index_col=[0])

In [225]:
# Create dictionaries that store information about the indicator-columns
academic_indicator_dict = {
    'unilist' : [x for x in data.columns if 'prob_to_read_share' in x],
    'litges' : [x for x in data.columns if 'page_count_rel_litges' in x],
    'vv' : [x for x in data.columns if 'event_count_rel_vv' in x],
    'lexica' : ['killy_length'],
    'editions' : ['reclam_count'],
    'bibliographies' : ['BDSL_hits_2000_all']
}

other_indicator_dict = {
    'staatsexamen' : ['staatsexamen_count'],
    'schullist' : [x for x in data.columns if 'schullist_' in x and 'share' in x],
    'abi' : ['abi_mentions'],
    'kanonspiel' : ['kanonspiel_points'],
    'segebrecht' : ['segebrecht_count'],
    'vv_hein' : ['vv_hein_count'],
    'wiki' : ['wiki_length_in_words']
}

all_indicator_dict = {**academic_indicator_dict, **other_indicator_dict}

all_indicator_label_dict = {
    'unilist' : 'Universitäre Leselisten',
    'litges' : 'Literaturgeschichten',
    'vv' : 'Universitäre Vorlesungsverzeichnisse',
    'lexica' : 'Fachlexika',
    'bibliographies' : 'Fachbibliographien',
    'editions' : 'Editionen',
    'vv_hein' : 'Vorlesungsverzeichnisse (Hein)',
    'staatsexamen' : 'Staatsexamina',
    'schullist' : 'Schulleselisten',
    'abi' : 'Abiturvorgaben',
    'wiki' : 'Online-Enzyklopädien',
    'kanonspiel' : 'Kanon-Spiel',
    'segebrecht' : 'Segebrecht'
}

In [226]:
# Statistics about indictors
print("Number of primary   indicators                         : ", len(all_indicator_dict.keys()))
print("Number of secondary indicators                         : ", \
      len([element for innerList in list(all_indicator_dict.values()) for element in innerList]))   

print("Number of primary   indicators for academic canonicity : ", len(academic_indicator_dict.keys()))
print("Number of secondary indicators for academic canonicity : ", \
      len([element for innerList in list(academic_indicator_dict.values()) for element in innerList]))      

Number of primary   indicators                         :  13
Number of secondary indicators                         :  69
Number of primary   indicators for academic canonicity :  6
Number of secondary indicators for academic canonicity :  50


## GND Filter

In [227]:
# Define relevant filters
relevant_occupations = [
    'Schriftsteller', 'Schriftstellerin', 'Dramatiker',
    'Lyriker', 'Drehbuchautor', 'Librettist',
    'Lyrikerin', 'Kirchenlieddichter',
    'Erzähler', 'Drehbuchautorin', 'Kinderbuchautor',
    'Dramatikerin', 'Librettistin', 'Liederdichter',
    'Kinderbuchautorin', 'Jugendbuchautor',
    'Minnesänger', 'Mundartschriftsteller', 'Jugendbuchautorin',
    'Musikschriftsteller', 'Romancier', 'Reiseschriftsteller',
    'Meistersinger', 'Kriminalschriftsteller', 'Liedermacher',
    'Prosaist', 'Kirchenlieddichterin', 'Prosaistin',
    'Theaterdichter', 'Heimatschriftsteller', 'Erzählerin',
    'Spruchdichter', 'Mundartschriftstellerin', 'Romanschriftstellerin',
    'Liederdichterin', 'Musikschriftstellerin',
    'Liedermacherin', 'Bilderbuchautor', 'Stadtschreiberin <Literatur>',
    'Kriminalschriftstellerin', 'Exilschriftsteller', 'Comicautorin',
    'Bestsellerautorin', 'Reiseschriftstellerin',

    'Satiriker', 'Hofdichter', 'Satirikerin',
    'Reisebuchautor',
]

relevant_countries = [
    'Deutschland',
    'Österreich',
    'Schweiz'
]

In [228]:
print("Number of people in dataset BEFORE gnd filtering : ", data.shape[0])

data = (
    data
    .query("GND_birth >= 1550")
    .loc[data['GND_occupation'].str.contains('|'.join(relevant_occupations), na=False)]
    .loc[data['GND_country'].str.contains('|'.join(relevant_countries), na=False)]
).copy()

print("Number of people in dataset AFTER  gnd filtering : ", data.shape[0])

Number of people in dataset BEFORE gnd filtering :  9628
Number of people in dataset AFTER  gnd filtering :  5209


## Manual Filter

In [229]:
entities_to_delete = [
    '118509039', # Benjamin, Walter
    '118594117', # Wagner, Richard
    '118500775', # Adorno, Theodor W.
    '118603426', # Rousseau, Jean-Jacques
    '118554727', # Humboldt, Wilhelm von
    '11862220X', # Thomasius, Christian
    '118607057', # Schelling, Friedrich Wilhelm Joseph von
    '118532847', # Fichte, Johann Gottlieb
    '118554700', # Humboldt, Alexander von
    '11850391X', # Arendt, Hannah
    '118638289', # Čechov, Anton Pavlovič
    '118627813', # Voltaire
    '118664123', # Blumenberg, Hans
    '118535749', # Friedrich II., Preußen, König
    '11574326X', # Arnold, Heinz Ludwig
    '118608045', # Schleiermacher, Friedrich
    '118525727', # Dilthey, Wilhelm
    '118599194', # Reich-Ranicki, Marcel
    '118584790', # Müller, Adam Heinrich
    '118556312', # Jacobi, Friedrich Heinrich
    '120309297', # Detering, Heinrich
    '118514881', # Breitinger, Johann Jakob
    '119090198', # Wagenseil, Johann Christoph
    '119505258', # Sepúlveda, Luis
    '118795163', # Schlosser, Johann Georg
    '118891871', # Skármeta, Antonio
    '121846067', # Illies, Florian
    '118630040', # Weidig, Friedrich Ludwig
    '118557327', # Jens, Walter
    '118631764', # Wessenberg, Ignaz Heinrich von
    '118516477', # Buber, Martin
    '11874559X', # Rochow, Friedrich Eberhard von
    '118628836', # Wallraff, Günter
    '118763784', # Unseld, Siegfried

    '118501259', # Ajtmatov, Čingiz
    '118513532', # Borges, Jorge Luis
    '118518208', # Byron, George Gordon Byron, Baron
    '118518739', # Camus, Albert
    '118519948', # Césaire, Aimé'
    '118541765', # Green, Julien
    '118555707', # Ionesco, Eugène
    '118765396', # Sillitoe, Alan
 ]

In [230]:
# Filter Entities from dataframe
print("Number of people in dataset BEFORE manual filtering : ", data.shape[0])
data = data[~data["GND"].isin(entities_to_delete)]

print("Number of people in dataset AFTER  manual filtering : ", data.shape[0])

Number of people in dataset BEFORE manual filtering :  5209
Number of people in dataset AFTER  manual filtering :  5167


In [231]:
scaler_0_1 = MinMaxScaler(feature_range=(0, 1))
scaler_1_1000 = MinMaxScaler(feature_range=(1, 1000))

def add_scores(
        df,
        all_indicator_dict = all_indicator_dict,
        canon_indicators = academic_indicator_dict.keys(),
        method_first_norm = 'zscore',
        method_first_merge = 'median',
        method_second_norm = 'zscore',
        method_second_merge = 'median'
        ):
    
    df = df.copy()
    
    # (1) apply zscore, scaling, and rank on secondary indicators
    for primary_indicator, secondary_indicators in all_indicator_dict.items():
        for secondary_indicator in secondary_indicators:
            df[secondary_indicator+'_zscore'] = zscore(df[secondary_indicator])
            df[secondary_indicator+'_scaled'] = scaler_0_1.fit_transform(df[[secondary_indicator]])
            df[secondary_indicator+'_rank'] = df[secondary_indicator].rank(method='min', ascending=False).astype(int)
            df = df.copy()

    # (2) create score for primary indicators by calculating mean/median of secondary indicators
    for primary_indicator, secondary_indicators in all_indicator_dict.items():
        if method_first_merge == 'mean':
            df[primary_indicator] = df[[x+'_'+method_first_norm for x in secondary_indicators]].mean(axis=1)
        elif method_first_merge == 'median':
            df[primary_indicator] = df[[x+'_'+method_first_norm for x in secondary_indicators]].median(axis=1)
        else:
            print("wrong method_first_merge")
    df = df.copy()

    # (3) apply zscore, scaling, rank, and relcount on primary indicators
    for primary_indicator, secondary_indicators in all_indicator_dict.items():
        df[primary_indicator+'_zscore'] = zscore(df[primary_indicator])
        df[primary_indicator+'_scaled'] = scaler_0_1.fit_transform(df[[primary_indicator]])
        df[primary_indicator+'_rank'] = df[primary_indicator].rank(method='min', ascending=False).astype(int)
        df[primary_indicator+'_relcount'] = (df[secondary_indicators] > 0).sum(axis=1)
    df = df.copy()

    # (4) create overall canonicity score by calculating mean/median of primary indicators
    if method_second_merge == 'mean':
        df['canonicity_score_raw'] = df[[x+'_'+method_second_norm for x in canon_indicators]].mean(axis=1)
    elif method_second_merge == 'median':
        df['canonicity_score_raw'] = df[[x+'_'+method_second_norm for x in canon_indicators]].median(axis=1)
    else:
        print("wrong method_second_merge")

    # (5) apply zscore, scaling, rank, and relcount on primary indicators
    df['canonicity_score_zscore'] = zscore(df[['canonicity_score_raw']])
    df['canonicity_score_scaled'] = scaler_1_1000.fit_transform(df[['canonicity_score_raw']])
    df['canonicity_score_rank'] = df['canonicity_score_raw'].rank(method='min', ascending=False).astype(int)
    df['canonicity_score_rank_cont'] = df['canonicity_score_raw'].rank(method='first', ascending=False).astype(int)
    df['canonicity_score_relcount'] = (df[[x+'_relcount' for x in academic_indicator_dict.keys()]] > 0).sum(axis=1)

    return df

In [232]:
data = add_scores(data).copy()

# Translate Gender data

In [233]:

terminology_map = {
    "Männlich": "male",
    "Weiblich": "female",
    "Unbekannt": "unknown",
    "Männlich, Weiblich": "male, female"
}
data["GND_gender"] = data["GND_gender"].apply(lambda x: terminology_map[x])


In [234]:
print("Absolute numbers our data gender: ")
print(data.groupby('GND_gender').size())

print("\n\n")

print("Relative numbers our data gender: ")
print(data["GND_gender"].value_counts(normalize=True))

Absolute numbers our data gender: 
GND_gender
female      879
male       4065
unknown     223
dtype: int64



Relative numbers our data gender: 
GND_gender
male       0.786723
female     0.170118
unknown    0.043159
Name: proportion, dtype: float64


## Clean time data in our data

In [235]:
# Birth year
correct_birth_years = []
correct_death_years = []

## birth year after 2020
for _, row in data.iterrows():
    birth_year = row["GND_birth"]
    if birth_year > 2020:
        print(f"ERROR: {row["GND_name"]} with id {row["GND"]} has birth year {birth_year}, corrected to NaN")
        correct_birth_years.append(np.nan)
    else:
        correct_birth_years.append(birth_year)

    death_year = row["GND_death"]
    if death_year < birth_year+1:
        print(f"ERROR: {row["GND_name"]} with id {row["GND"]} has death_year ({death_year}) before birth_year ({birth_year}), corrected to NaN")
        correct_death_years.append(np.nan)
    # Grob Autoren rausfiltern, bei denen Todesjahr zu hoch
    elif death_year > birth_year+120:
        print(f"ERROR: {row["GND_name"]} with id {row["GND"]} has death ({death_year}) year more than 120 years after birth year ({birth_year}), corrected to NaN")
        correct_death_years.append(np.nan)
    else:
        correct_death_years.append(death_year)

data["GND_birth"] = correct_birth_years
data["GND_death"] = correct_death_years

ERROR: Lindemayr, Maurus with id 100191290 has death (1983.0) year more than 120 years after birth year (1723.0), corrected to NaN
ERROR: Meißner, August Gottlieb with id 100308023 has death (1907.0) year more than 120 years after birth year (1753.0), corrected to NaN
ERROR: Ast, Friedrich with id 118650742 has death_year (1778.0) before birth_year (1778.0), corrected to NaN
ERROR: Frischauer, Paul with id 119510286 has death_year (1777.0) before birth_year (1898.0), corrected to NaN
ERROR: Schallenberg, Christoph von with id 124569382 has death_year (1597.0) before birth_year (1597.0), corrected to NaN


# Import GND data

In [236]:
gnd_df = pd.read_csv("../data/gnd_all.csv", sep=";", encoding="utf-8", index_col=0)

# Apply manual filter
gnd_df = gnd_df[~gnd_df["gndIdentifier"].isin(entities_to_delete)]

# Translate gender labels
gnd_df["GND_gender"] = gnd_df["GND_gender"].apply(lambda x: terminology_map[x])

print("Absolute numbers GND gender: ")
print(gnd_df.groupby('GND_gender').size())

print("\n\n")

print("Relative numbers GND gender: ")
print(gnd_df["GND_gender"].value_counts(normalize=True))


Absolute numbers GND gender: 
GND_gender
female          19582
male            47941
male, female        1
unknown         11443
dtype: int64



Relative numbers GND gender: 
GND_gender
male            0.607102
female          0.247977
unknown         0.144909
male, female    0.000013
Name: proportion, dtype: float64


In [237]:
fig = px.scatter(gnd_df, x="GND_birth", y="GND_death", hover_data=["GND_name"])
fig.show()

## Investigate Unknown

In [238]:
# define subset for investigation
gnd_df_subset = gnd_df.copy() # .query("GND_birth >= 1950")

display(gnd_df_subset.shape[0])
display(gnd_df_subset['GND_gender'].value_counts(normalize=False))

78967

GND_gender
male            47941
female          19582
unknown         11443
male, female        1
Name: count, dtype: int64

In [239]:
# enrich subset: extract first name and create binary variable 'female'
def extract_first_name(name):
    name_split = name.split(', ')
    if len(name_split) >= 2:
        first_name = name_split[-1]
        if first_name.endswith(' von'):
            first_name = first_name[:-4]
        if first_name.endswith(' zu'):
            first_name = first_name[:-3]
        if first_name.endswith(' de'):
            first_name = first_name[:-3]
        return first_name.strip()
    else:
        return float('NaN')

gnd_df_subset['first_name'] = gnd_df_subset['GND_name'].apply(extract_first_name)
gnd_df_subset['GND_female'] = [1 if 'female' in x else float('NaN') if 'unknown' in x else 0 for x in gnd_df_subset['GND_gender']]

gnd_df_subset.head()

,gndIdentifier,GND_name,GND_occupation,GND_gender,GND_country,placeOfBirth,GND_birth,GND_death,periodOfActivity,first_name,GND_female
0,1084026554,"Kuti, Michaela","Sängerin, Musikerin, Gesangslehrerin, Liederma...",female,Deutschland,Sindelfingen,1978.0,NaN,NaN,Michaela,1.0
1,1106279247,"Rabhansl, Karin","Musikerin, Sängerin, Gitarristin, Komponistin,...",female,Deutschland,Hutthurm,1986.0,NaN,NaN,Karin,1.0
2,1148348581,"Kükenshöner, Bärbel","Musiktherapeutin, Liedermacherin",female,Deutschland,NaN,1974.0,NaN,NaN,Bärbel,1.0
3,118004034,"Wegner, Bettina","Liedermacherin, Komponistin, Musikerin, Sänger...",female,Deutschland,Berlin-Lichterfelde,1947.0,NaN,NaN,Bettina,1.0
4,132979055,"Schwab, Stefanie","Pädagogin, Liedermacherin, Sängerin",female,Deutschland,NaN,1963.0,NaN,NaN,Stefanie,1.0


In [240]:
# create gender_table based on known GND gender labels in subset
gnd_group = gnd_df_subset.query("GND_gender != 'unknown'").groupby('first_name')
gender_df = gnd_group.size().to_frame('known_count')
gender_df['female_count']  = gnd_group['GND_female'].sum()
gender_df['female_prob'] = gender_df['female_count']/gender_df['known_count']

display(gender_df.loc[['Thomas', 'Marie', 'Yvonne']])

# example for misclassification: first name is 'Yvonne', yet GND gender is 'male'
display(gnd_df_subset.query("first_name=='Yvonne' and GND_gender == 'male'"))
display(gnd_df_subset.query("first_name=='Thomas' and GND_gender == 'female'"))

,known_count,female_count,female_prob
first_name,,,
Thomas,398,1.0,0.002513
Marie,206,204.0,0.990291
Yvonne,24,23.0,0.958333


,gndIdentifier,GND_name,GND_occupation,GND_gender,GND_country,placeOfBirth,GND_birth,GND_death,periodOfActivity,first_name,GND_female
4580,1047823993,"Teichmann, Yvonne","Reiseverkehrskauffrau, Lyrikerin",male,Deutschland,NaN,1979.0,NaN,NaN,Yvonne,0.0


,gndIdentifier,GND_name,GND_occupation,GND_gender,GND_country,placeOfBirth,GND_birth,GND_death,periodOfActivity,first_name,GND_female
9581,1348632313,"Schmid, Thomas","Künstler, Lyriker",female,Deutschland,Leipzig,1952.0,NaN,NaN,Thomas,1.0


In [241]:
# predict gender based on gender_table for the subset
def predict_gender (first_name):
    if first_name in gender_df.index:
        predicted_gender = gender_df.loc[first_name, 'female_prob']
        prediction_support = gender_df.loc[first_name, 'known_count']
    else:
        predicted_gender = float('NaN')
        prediction_support = float('NaN')
    return (predicted_gender, prediction_support)

gnd_df_subset[['female_prob_predicted', 'prediction_support']] = gnd_df_subset['first_name'].apply(predict_gender).apply(pd.Series)
gnd_df_subset['Pred_gender'] = ['female' if x >= 0.9 else 'male' if x <= 0.1 else 'unknown' for x in gnd_df_subset['female_prob_predicted']]

gnd_df_subset.query("GND_gender == 'unknown'").sample(n=5)

,gndIdentifier,GND_name,GND_occupation,GND_gender,GND_country,placeOfBirth,GND_birth,GND_death,periodOfActivity,first_name,GND_female,female_prob_predicted,prediction_support,Pred_gender
52845,1131590333,"Oden, Matthias","Redakteur, Werbefachmann, Schriftsteller",unknown,Deutschland,NaN,1979.0,NaN,NaN,Matthias,NaN,0.0,176.0,male
43318,118800450,"Hofmann, Murad Wilfried","Diplomat, Jurist, Schriftsteller",unknown,Deutschland,Aschaffenburg,1931.0,2020.0,NaN,Murad Wilfried,NaN,NaN,NaN,unknown
64173,128549211,"Herrmann, Fritz","Schriftsteller, Journalist, Dramatiker, Satiri...",unknown,Österreich,NaN,1922.0,2003.0,NaN,Fritz,NaN,0.0,415.0,male
71477,128088389,"Podzeit, Wulf","Archäologe, Filmproduzent, Journalist, Schrift...",unknown,Österreich,Salzburg,1940.0,2009.0,NaN,Wulf,NaN,0.0,10.0,male
38319,129002372,"Jolles, André G.","Schriftsteller, Komponist",unknown,Deutschland,NaN,1970.0,NaN,NaN,André G.,NaN,NaN,NaN,unknown


In [242]:
# the final result: predicted gender distribution in 'unknown' gender vs. gender distribution in known gender
display(gnd_df_subset.query("GND_gender == 'unknown'")['Pred_gender'].value_counts(normalize=False))
display(gnd_df_subset.query("GND_gender != 'unknown'")['GND_gender'].value_counts(normalize=False))

Pred_gender
male       6393
female     2995
unknown    2055
Name: count, dtype: int64

GND_gender
male            47941
female          19582
male, female        1
Name: count, dtype: int64

In [243]:
known = gnd_df_subset[gnd_df_subset['GND_gender'] != 'unknown']['GND_gender'].value_counts()
predicted = gnd_df_subset.query("GND_gender == 'unknown' and Pred_gender != 'unknown'")['Pred_gender'].value_counts()

table = [
    [known['female'], known['male']],
    [predicted['female'], predicted['male']]
]

chi2, p, dof, expected = chi2_contingency(table)

print(f"chi2: {chi2:.4f}")
print(f"p   : {p:.4e}")

chi2: 33.3299
p   : 7.7776e-09


# General definitions for all plots

In [244]:
# Sort our data by canonicity score
data = data.sort_values(by='canonicity_score_raw', ascending=False)

# Define Color Map for all Plots
color_map = {
    "male": "sandybrown",
    "female": "maroon",
    "unknown": "lightseagreen",
    "male, female": "lightskyblue"
}



# Distribution: Gender and canon score 

In [245]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.05
)

# Top 500

df500 = data.head(500)

fig.add_trace(
    go.Bar(
        x=df500["GND_name"],
        y=df500["canonicity_score_scaled"],
        marker_color=df500["GND_gender"].map(color_map),
        showlegend=False
    ),
    row=1, col=1
)


# Top 50

df50 = data.head(50)

fig.add_trace(
    go.Bar(
        x=df50["GND_name"],
        y=df50["canonicity_score_scaled"],
        marker_color=df50["GND_gender"].map(color_map),
        showlegend=False
    ),
    row=2, col=1
)


# Legend

for gender_value, color in color_map.items():
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode='markers',
            marker=dict(color=color, size=12),
            name=gender_value, 
            showlegend=True
        ),
        row=1, col=1
    )

# Layout
fig.update_layout(
    height=800,
    legend_title_text="Gender (as in GND)",
    showlegend=True
)

# Y-Axes
fig.update_yaxes(title_text="Canonicity Score (scaled)", row=1, col=1)
fig.update_yaxes(title_text="Canonicity Score (scaled)", row=2, col=1)

# X-Axes
fig.update_xaxes(showticklabels=False, row=1, col=1)

# lower x axes
fig.update_xaxes(
    showticklabels=True,
    tickfont=dict(size=12),
    categoryorder="total descending",
    title_text="Authors",
    row=2, col=1
)

fig.show()


# Gender and Rank/Canonicity Score

In [246]:
# Wie viele Autorinnen sind unter den Top 1, 5, 100 usw.
data = data.sort_values(by='canonicity_score_raw', ascending=False).reset_index(drop=True)

shares_per_rank = {"Rank": [], "male": [], "female": [], "unknown": []}



for n_top in range(1, len(data)+1):
    shares_per_rank["Rank"].append(n_top)
    top_data = data.head(n_top)
    value_counts = top_data["GND_gender"].value_counts()
    for parameter in ["male", "female", "unknown"]:
        if parameter in value_counts.index: # Kommt Männlich, Weiblich usw. überhaupt vor?
            shares_per_rank[parameter].append(value_counts[parameter]/n_top)
        else:
            shares_per_rank[parameter].append(0)
            

result = pd.DataFrame(shares_per_rank)
#result

In [247]:
fig = go.Figure()

for col in ["female", "unknown", "male"]:
    fig.add_trace(go.Scatter(
        x=result["Rank"],
        y=result[col],
        mode="lines",
        line=dict(width=0.5, color=color_map[col]),
        stackgroup="one",
        name=col
    ))


# Baseline 1: horizontal Line at 50%

fig.add_hline(
    y=0.5,
    line_width=1,
    line_dash="dash",
    line_color="black"
)

## Dummy-Trace for Legend
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode="lines",
    line=dict(width=1, dash="dash", color="black"),
    name="50 % baseline",
    showlegend=True
))


# Baseline 2: Female Share in the GND

anteil_female_gnd = gnd_df["GND_gender"].value_counts(normalize=True)["female"]

fig.add_hline(
    y=anteil_female_gnd,
    line_width=2,
    line_dash="dot",
    line_color="black"
)

## Dummy-Trace for Legend
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode="lines",
    line=dict(width=2, dash="dot", color="black"),
    name="Female share (GND)",
    showlegend=True
))


# Layout

fig.update_layout(
    yaxis=dict(title="Share", tickformat=".0%", range=[0, 1]),
    xaxis=dict(title="Rank"),
    template="plotly_white",
    legend_title_text="Legend",
    showlegend=True,
    height=600
)

fig.show()


# Stacked Area Plots: Birth year in our data and the GND

In [248]:
df = gnd_df.copy()
df = df[df["GND_birth"]<2010]

# Sum per Year and Gender
counts = (
    df.groupby(["GND_birth", "GND_gender"])
      .size()
      .reset_index(name="count")
)

# Total amount per Year
counts["total"] = counts.groupby("GND_birth")["count"].transform("sum")

# Gender Share
counts["share"] = counts["count"] / counts["total"]

# Pivot
pivot = counts.pivot(index="GND_birth", columns="GND_gender", values="share").fillna(0)

pivot = pivot[["female", "male, female", "unknown", "male"]]

fig = px.area(
    pivot,
    x=pivot.index,
    y=pivot.columns,
    color_discrete_map=color_map,
)

# horizontal Line at 50
fig.add_hline(
    y=0.5,
    line_width=1,
    line_dash="dash",
    line_color="black"
)

## Dummy-Trace for Legend
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode="lines",
    line=dict(width=1, dash="dash", color="black"),
    name="50 % baseline",
    showlegend=True
))

fig.update_layout(
    xaxis_title="Year of Birth",
    yaxis_title="Share",
    legend_title_text="Legend",
)

fig.show()

In [249]:
# Baseline: Female Share per year in gnd

df_all = gnd_df.copy()
df_all = df_all[df_all["GND_birth"]<2010]

counts_all = (
    df_all.groupby(["GND_birth", "GND_gender"])
          .size()
          .reset_index(name="count")
)

counts_all["total"] = counts_all.groupby("GND_birth")["count"].transform("sum")
counts_all["share"] = counts_all["count"] / counts_all["total"]

pivot_all = counts_all.pivot(index="GND_birth", columns="GND_gender", values="share").fillna(0)

baseline_female = pivot_all["female"]

baseline_female_smooth = (
    baseline_female
    .rolling(window=8, min_periods=1, center=True)
    .mean()
)

In [250]:
color_map_subplots = {
    "male": "rgba(244, 198, 157, 0.9)",
    "female": "rgba(178, 118, 127, 0.9)",
    "unknown": "rgba(123,207,210, 0.9)",
    "male, female": "rgba(198, 225, 246, 0.9)"
}

# Define function
def prepare_pivot(df):
    counts = (
        df.groupby(["GND_birth", "GND_gender"])
          .size()
          .reset_index(name="count")
    )

    counts["total"] = counts.groupby("GND_birth")["count"].transform("sum")
    counts["share"] = counts["count"] / counts["total"]

    pivot = counts.pivot(index="GND_birth", columns="GND_gender", values="share").fillna(0)

    # Ordnung + smoothing
    cols = [c for c in ["female", "unknown", "male"] if c in pivot.columns]
    pivot = pivot[cols]
    pivot_smooth = pivot.rolling(window=8, min_periods=1, center=True).mean()

    return pivot_smooth


# Subsets
datasets = [
    ("All", data.head(6000)),
    ("Top 500",  data.head(500)),
    ("Top 100",  data.head(100)),
    ("Top 50",   data.head(50))
]

# Create Subplots
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=[name for name, _ in datasets]
)

# for every top n
for i, (title, df_subset) in enumerate(datasets, start=1):
    pivot = prepare_pivot(df_subset)

    # Stacked Areas
    for gender in pivot.columns:
        fig.add_trace(
            go.Scatter(
                x=pivot.index,
                y=pivot[gender],
                stackgroup="one",
                mode="none",
                name=gender,
                fillcolor=color_map_subplots[gender],
                showlegend=True if i == 1 else False
            ),
            row=i, col=1
        )

    # 50%-Line
    fig.add_hline(
        y=0.5,
        line_width=1,
        line_dash="dash",
        line_color="black",
        row=i,
        col=1
    )

    # Line Female Share overall
    fig.add_scatter(
        x=baseline_female_smooth.index,
        y=baseline_female_smooth.values,
        mode="lines",
        line=dict(color="black", width=2, dash="dot"),
        showlegend=False,
        row=i,
        col=1
    )

# Dummy-Traces for Legend

# 50%
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="lines",
    line=dict(width=1, dash="dash", color="black"),
    name="50 % baseline",
    showlegend=True
))

# Overall female share
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="lines",
    line=dict(width=2, dash="dot", color="black"),
    name="Smoothed female share (GND)",
    showlegend=True
))


# Layout - Achsen

# Y-Axes
for r in range(1, 5):
    fig.update_yaxes(title_text="Share (smoothed by rolling avg 8)", row=r, col=1)

# X-Axes
fig.update_xaxes(title_text="Year of Birth", row=4, col=1)

for r in range(1, 4):
    fig.update_xaxes(title_text="", row=r, col=1)

fig.update_layout(
    height=1450,
    showlegend=True,
    legend_title_text="Legend",
)

fig.show()


# Additional Analyses

In [251]:
data['century'] = (data['GND_birth'] // 100) * 100
top_ranks = [99999999, 1000, 900, 800, 700, 600, 500, 400, 300, 200, 100]

results = []

for century in sorted(data['century'].unique()):
    data_cent = data[data['century'] == century].query("GND_gender == 'male' or GND_gender == 'female'")
    
    for top in top_ranks:
        data_top = data_cent.nsmallest(top, 'canonicity_score_rank')
        
        all_count = data_top.shape[0]
        male_count = (data_top['GND_gender'] == 'male').sum()
        female_count = (data_top['GND_gender'] == 'female').sum()
        
        results.append({
            'century': century,
            'top': str(top),
            'all_count': all_count,
            'male_count': male_count,
            'female_count': female_count
        })

results_df = pd.DataFrame(results)
results_df['male_share'] = results_df['male_count'] / results_df['all_count']
results_df['female_share'] = results_df['female_count'] / results_df['all_count']

fig = px.bar(
    results_df,
    x='century',
    y='female_share',
    color='top',
    barmode='group',
    labels={'female_share': 'Share of Female Authors', 'century': 'Century'},
    title='Female Share in Top-N Gender-Known Authors by Century'
)

fig.show()